# Restriction Enzyme 3mer AA Analysis

This notebook analyzes Site I and Site II restriction enzymes, calculating:
1. **Summary statistics**: For each RE, count unique 3mer AAs and how many support silent mutations
2. **Detailed information**: Complete 3mer AA data with optimal codons and mutation sequences
3. **Individual exports**: Save per-enzyme CSV files and a comprehensive master file

## Step 1: Import Libraries and Load Data

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import os
from Bio.Restriction import Restriction
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("RESTRICTION ENZYME 3MER AA ANALYSIS")
print("="*80)

# Load the pre-computed data
print("\n1. Loading enzyme data...")
df_seamless_insert = pd.read_csv('./utils/output/restriction_enzyme_seamless_insert.csv')
df_silent_mutation = pd.read_csv('./utils/output/restriction_enzyme_slient_mutation.csv')

print(f"   ✓ Loaded seamless insert data: {len(df_seamless_insert):,} rows")
print(f"   ✓ Loaded silent mutation data: {len(df_silent_mutation):,} rows")
print(f"   ✓ Seamless insert enzymes: {df_seamless_insert['name'].nunique()}")
print(f"   ✓ Silent mutation enzymes: {df_silent_mutation['name'].nunique()}")

RESTRICTION ENZYME 3MER AA ANALYSIS

1. Loading enzyme data...
   ✓ Loaded seamless insert data: 14,180 rows
   ✓ Loaded silent mutation data: 7,727 rows
   ✓ Seamless insert enzymes: 70
   ✓ Silent mutation enzymes: 57


## Step 2: Load Pre-selected Enzymes (Site I and Site II)

Load the curated enzyme lists from `enzyme_selection_analysis.ipynb` output.
These are the 47 selected Site I enzymes and Site II enzymes that meet quality criteria.

In [2]:
print("\n2. Loading pre-selected enzymes from enzyme_selection_analysis.ipynb...")

# Load curated enzyme lists (47 Site I, Site II subset, and Site III)
df_selected_site_i = pd.read_csv('./output/selected_site_i_enzymes.csv')
df_selected_site_ii = pd.read_csv('./output/selected_site_ii_enzymes.csv')

print(f"   ✓ Loaded selected Site I enzymes: {len(df_selected_site_i)}")
print(f"   ✓ Loaded selected Site II enzymes: {len(df_selected_site_ii)}")

# Get enzyme lists
selected_site_i_enzymes = sorted(df_selected_site_i['enzyme'].unique())
selected_site_ii_enzymes = sorted(df_selected_site_ii['enzyme'].unique())

print(f"   ✓ Unique Site I enzymes: {len(selected_site_i_enzymes)}")
print(f"   ✓ Unique Site II enzymes: {len(selected_site_ii_enzymes)}")
print(f"   ✓ Site II ⊂ Site I: {set(selected_site_ii_enzymes).issubset(set(selected_site_i_enzymes))}")

# Get union of selected enzymes (Site I is superset, so it's the union)
all_enzymes = selected_site_i_enzymes

print(f"\n   Filter criteria applied:")
print(f"   - Site I: Regular enzymes, methylation insensitive, any overhang")
print(f"   - Site II: Regular enzymes, methylation insensitive, overhang in [-4, -3, +2]")

# Helper function to get enzyme recognition site
def get_recognition_site(enzyme_name):
    """Get recognition site for an enzyme"""
    try:
        enzyme = getattr(Restriction, enzyme_name)
        return str(enzyme.site)
    except:
        return "N/A"


2. Loading pre-selected enzymes from enzyme_selection_analysis.ipynb...
   ✓ Loaded selected Site I enzymes: 47
   ✓ Loaded selected Site II enzymes: 30
   ✓ Unique Site I enzymes: 47
   ✓ Unique Site II enzymes: 30
   ✓ Site II ⊂ Site I: True

   Filter criteria applied:
   - Site I: Regular enzymes, methylation insensitive, any overhang
   - Site II: Regular enzymes, methylation insensitive, overhang in [-4, -3, +2]


## Step 3: Calculate Summary Statistics per Enzyme

For each **selected** enzyme (47 total), count:
- Total unique 3mer AAs it can insert
- How many of those 3mer AAs can introduce silent mutations

In [3]:
print("\n3. Calculating summary statistics per enzyme...")

summary_stats = []

for enzyme in all_enzymes:
    # Get 3mer AAs from seamless insert (Site I)
    site_i_data = df_seamless_insert[df_seamless_insert['name'] == enzyme]
    site_i_3mers = set(site_i_data['re_site_shifted_tl'].unique())
    
    # Get 3mer AAs from silent mutation (Site II)
    site_ii_data = df_silent_mutation[df_silent_mutation['name'] == enzyme]
    site_ii_3mers = set(site_ii_data['re_site_shifted_tl'].unique())
    
    # Total unique 3mer AAs
    all_3mers = site_i_3mers | site_ii_3mers
    
    # 3mer AAs that support silent mutation
    silent_mutation_3mers = site_ii_3mers
    
    # Get recognition site
    rec_site = get_recognition_site(enzyme)
    
    summary_stats.append({
        'enzyme': enzyme,
        'recognition_site': rec_site,
        'total_unique_3mer_aa': len(all_3mers),
        'silent_mutation_3mer_aa': len(silent_mutation_3mers),
        'seamless_only_3mer_aa': len(site_i_3mers - site_ii_3mers)
    })

df_summary = pd.DataFrame(summary_stats)

print(f"   ✓ Summary calculated for {len(df_summary)} enzymes")
print(f"\n   Sample statistics:")
print(f"   - Average unique 3mer AAs per enzyme: {df_summary['total_unique_3mer_aa'].mean():.1f}")
print(f"   - Average silent mutation 3mer AAs: {df_summary['silent_mutation_3mer_aa'].mean():.1f}")
print(f"   - Silent mutation coverage: {df_summary['silent_mutation_3mer_aa'].sum() / df_summary['total_unique_3mer_aa'].sum() * 100:.1f}%")

print("\nTop 10 enzymes by total 3mer AA coverage:")
display(df_summary.nlargest(10, 'total_unique_3mer_aa')[['enzyme', 'recognition_site', 'total_unique_3mer_aa', 'silent_mutation_3mer_aa']])


3. Calculating summary statistics per enzyme...
   ✓ Summary calculated for 47 enzymes

   Sample statistics:
   - Average unique 3mer AAs per enzyme: 179.2
   - Average silent mutation 3mer AAs: 99.7
   - Silent mutation coverage: 55.6%

Top 10 enzymes by total 3mer AA coverage:


,enzyme,recognition_site,total_unique_3mer_aa,silent_mutation_3mer_aa
1,AciI,CCGC,1354,819
44,TaqI,TCGA,1129,585
33,NlaIII,CATG,1110,0
24,HpaII,CCGG,897,585
22,HhaI,GCGC,791,546
28,MseI,TTAA,518,280
11,BseYI,CCCAGC,148,114
14,BspEI,TCCGGA,98,74
18,BstBI,TTCGAA,98,98
16,BsrGI,TGTACA,91,73


## Step 4: Generate Detailed 3mer AA Information

Create comprehensive dataframe with:
- RE name
- Recognition site  
- 3mer AA
- Optimal 9bp sequence (original)
- Whether silent mutation is possible
- Optimal 9bp after mutation (if applicable)
- Codon usage frequency

In [4]:
print("\n4. Generating detailed 3mer AA information...")

detailed_data = []

for enzyme in all_enzymes:
    rec_site = get_recognition_site(enzyme)
    
    # Process Site I data (seamless insert - can always use original sequence)
    site_i_data = df_seamless_insert[df_seamless_insert['name'] == enzyme].copy()
    
    # Keep best codon usage for each 3mer AA
    site_i_best = site_i_data.sort_values('codon_usage', ascending=False).groupby('re_site_shifted_tl').first().reset_index()
    
    # Process Site II data (silent mutation - has mutated sequences)
    site_ii_data = df_silent_mutation[df_silent_mutation['name'] == enzyme].copy()
    site_ii_best = site_ii_data.sort_values('codon_usage_mutate', ascending=False).groupby('re_site_shifted_tl').first().reset_index()
    
    # Create a mapping of 3mer AA -> silent mutation info
    silent_mutation_map = {}
    for _, row in site_ii_best.iterrows():
        silent_mutation_map[row['re_site_shifted_tl']] = {
            'dna_seq_mutated': row['re_site_mutate_shifted'],
            'codon_usage_mutate': row['codon_usage_mutate']
        }
    
    # Process all 3mer AAs for this enzyme
    for _, row in site_i_best.iterrows():
        threemer_aa = row['re_site_shifted_tl']
        
        # Check if silent mutation is possible for this 3mer
        can_silent_mutation = threemer_aa in silent_mutation_map
        
        detail_entry = {
            'enzyme': enzyme,
            'recognition_site': rec_site,
            '3mer_aa': threemer_aa,
            'optimal_9bp_original': row['re_site_shifted'],
            'codon_usage_original': row['codon_usage'],
            'can_silent_mutation': can_silent_mutation,
            'optimal_9bp_mutated': silent_mutation_map[threemer_aa]['dna_seq_mutated'] if can_silent_mutation else None,
            'codon_usage_mutated': silent_mutation_map[threemer_aa]['codon_usage_mutate'] if can_silent_mutation else None
        }
        
        detailed_data.append(detail_entry)

df_detailed = pd.DataFrame(detailed_data)

print(f"   ✓ Generated {len(df_detailed):,} detailed entries")
print(f"   ✓ Unique enzymes: {df_detailed['enzyme'].nunique()}")
print(f"   ✓ Unique 3mer AAs: {df_detailed['3mer_aa'].nunique()}")
print(f"   ✓ Entries with silent mutation: {df_detailed['can_silent_mutation'].sum():,} ({df_detailed['can_silent_mutation'].sum()/len(df_detailed)*100:.1f}%)")

print("\n   Sample entries:")
display(df_detailed.head(10))


4. Generating detailed 3mer AA information...
   ✓ Generated 8,424 detailed entries
   ✓ Unique enzymes: 45
   ✓ Unique 3mer AAs: 4229
   ✓ Entries with silent mutation: 4,684 (55.6%)

   Sample entries:


,enzyme,recognition_site,3mer_aa,optimal_9bp_original,codon_usage_original,can_silent_mutation,optimal_9bp_mutated,codon_usage_mutated
0,AatII,GACGTC,ADV,GCGGACGTC,23.566667,False,None,NaN
1,AatII,GACGTC,ATS,GCGACGTCG,19.333333,False,None,NaN
2,AatII,GACGTC,CDV,TGCGACGTC,13.400000,False,None,NaN
3,AatII,GACGTC,DDV,GATGACGTC,23.366667,False,None,NaN
4,AatII,GACGTC,DVA,GACGTCGCG,23.566667,False,None,NaN
5,AatII,GACGTC,DVC,GACGTCTGC,13.400000,False,None,NaN
6,AatII,GACGTC,DVD,GACGTCGAT,23.366667,False,None,NaN
7,AatII,GACGTC,DVE,GACGTCGAA,25.300000,False,None,NaN
8,AatII,GACGTC,DVF,GACGTCTTT,17.300000,False,None,NaN
9,AatII,GACGTC,DVG,GACGTCGGC,21.866667,False,None,NaN


## Step 5: Export Individual Enzyme CSV Files

Save detailed information for each enzyme to separate CSV files.

In [5]:
print("\n5. Exporting individual enzyme CSV files...")

# Create output directory for individual enzyme files
output_dir = './output/re_3mer_individual'
os.makedirs(output_dir, exist_ok=True)

# Export each enzyme's data to a separate CSV
for enzyme in all_enzymes:
    enzyme_data = df_detailed[df_detailed['enzyme'] == enzyme]
    output_path = os.path.join(output_dir, f'{enzyme}_3mer_details.csv')
    enzyme_data.to_csv(output_path, index=False)

print(f"   ✓ Exported {len(all_enzymes)} individual CSV files to: {output_dir}/")
print(f"   Example files:")
for i, enzyme in enumerate(all_enzymes[:5]):
    print(f"     - {enzyme}_3mer_details.csv ({len(df_detailed[df_detailed['enzyme'] == enzyme])} rows)")


5. Exporting individual enzyme CSV files...
   ✓ Exported 47 individual CSV files to: ./output/re_3mer_individual/
   Example files:
     - AatII_3mer_details.csv (63 rows)
     - AciI_3mer_details.csv (1354 rows)
     - AclI_3mer_details.csv (82 rows)
     - AflII_3mer_details.csv (70 rows)
     - AgeI_3mer_details.csv (76 rows)


## Step 6: Export Complete Summary and Detailed CSV Files

In [6]:
print("\n6. Exporting complete CSV files...")

# Export summary statistics
summary_output_path = './output/re_3mer_summary_statistics.csv'
df_summary.to_csv(summary_output_path, index=False)
print(f"   ✓ Summary statistics saved to: {summary_output_path}")
print(f"     ({len(df_summary)} enzymes)")

# Export complete detailed information
detailed_output_path = './output/re_3mer_detailed_complete.csv'
df_detailed.to_csv(detailed_output_path, index=False)
print(f"   ✓ Complete detailed data saved to: {detailed_output_path}")
print(f"     ({len(df_detailed):,} rows)")

print("\n" + "="*80)
print("✓ ANALYSIS COMPLETE")
print("="*80)

print("\n📊 Summary:")
print(f"   - Total enzymes analyzed: {len(all_enzymes)}")
print(f"   - Total unique 3mer AAs: {df_detailed['3mer_aa'].nunique()}")
print(f"   - Total enzyme-3mer combinations: {len(df_detailed):,}")
print(f"   - Combinations with silent mutation: {df_detailed['can_silent_mutation'].sum():,}")

print("\n📁 Output files:")
print(f"   1. Summary: {summary_output_path}")
print(f"   2. Detailed: {detailed_output_path}")
print(f"   3. Individual: {output_dir}/*.csv ({len(all_enzymes)} files)")


6. Exporting complete CSV files...
   ✓ Summary statistics saved to: ./output/re_3mer_summary_statistics.csv
     (47 enzymes)
   ✓ Complete detailed data saved to: ./output/re_3mer_detailed_complete.csv
     (8,424 rows)

✓ ANALYSIS COMPLETE

📊 Summary:
   - Total enzymes analyzed: 47
   - Total unique 3mer AAs: 4229
   - Total enzyme-3mer combinations: 8,424
   - Combinations with silent mutation: 4,684

📁 Output files:
   1. Summary: ./output/re_3mer_summary_statistics.csv
   2. Detailed: ./output/re_3mer_detailed_complete.csv
   3. Individual: ./output/re_3mer_individual/*.csv (47 files)


## Verification and Sample Display

Display samples from the generated data to verify correctness.

In [7]:
print("\n" + "="*80)
print("VERIFICATION AND SAMPLES")
print("="*80)

print("\n1. Summary statistics sample:")
display(df_summary.head(10))

print("\n2. Detailed data sample (with silent mutations):")
df_with_mutation = df_detailed[df_detailed['can_silent_mutation'] == True]
display(df_with_mutation.head(10))

print("\n3. Detailed data sample (without silent mutations):")
df_without_mutation = df_detailed[df_detailed['can_silent_mutation'] == False]
display(df_without_mutation.head(10))

print("\n4. Example: Check one enzyme in detail")
example_enzyme = all_enzymes[0]
print(f"   Enzyme: {example_enzyme}")
example_data = df_detailed[df_detailed['enzyme'] == example_enzyme]
print(f"   Total 3mer AAs: {len(example_data)}")
print(f"   With silent mutation: {example_data['can_silent_mutation'].sum()}")
display(example_data)


VERIFICATION AND SAMPLES

1. Summary statistics sample:


,enzyme,recognition_site,total_unique_3mer_aa,silent_mutation_3mer_aa,seamless_only_3mer_aa
0,AatII,GACGTC,63,0,63
1,AciI,CCGC,1354,819,535
2,AclI,AACGTT,82,82,0
3,AflII,CTTAAG,70,70,0
4,AgeI,ACCGGT,76,56,20
5,ApaI,GGGCCC,55,0,55
6,ApaLI,GTGCAC,73,59,14
7,AscI,GGCGCGCC,4,3,1
8,AvrII,CCTAGG,55,55,0
9,BclI,TGATCA,51,33,18



2. Detailed data sample (with silent mutations):


,enzyme,recognition_site,3mer_aa,optimal_9bp_original,codon_usage_original,can_silent_mutation,optimal_9bp_mutated,codon_usage_mutated
77,AciI,CCGC,AAR,GCGGCGCGC,34.333333,True,GCGGCGCGC,34.333333
84,AciI,CCGC,ACR,GCGTGCCGC,24.166667,True,GCGTGTCGC,23.466667
99,AciI,CCGC,ADR,GCGGATCGC,34.133333,True,GCGGATCGC,34.133333
119,AciI,CCGC,AER,GCGGAACGC,36.066667,True,GCGGAACGG,28.766667
125,AciI,CCGC,AFR,GCGTTCCGC,26.500000,True,GCGTTTCGC,28.066667
140,AciI,CCGC,AGR,GCGGGCCGC,32.633333,True,GCGGGCCGT,31.000000
146,AciI,CCGC,AHR,GCGCACCGC,25.866667,True,GCGCATCGC,26.766667
147,AciI,CCGC,AIR,GCGATCCGC,27.566667,True,GCGATTCGC,31.666667
148,AciI,CCGC,AKR,GCGAAGCGG,18.233333,True,GCGAAGCGC,25.533333
149,AciI,CCGC,ALR,GCGCTGCGG,29.833333,True,GCGCTGCGC,37.133333



3. Detailed data sample (without silent mutations):


,enzyme,recognition_site,3mer_aa,optimal_9bp_original,codon_usage_original,can_silent_mutation,optimal_9bp_mutated,codon_usage_mutated
0,AatII,GACGTC,ADV,GCGGACGTC,23.566667,False,None,NaN
1,AatII,GACGTC,ATS,GCGACGTCG,19.333333,False,None,NaN
2,AatII,GACGTC,CDV,TGCGACGTC,13.400000,False,None,NaN
3,AatII,GACGTC,DDV,GATGACGTC,23.366667,False,None,NaN
4,AatII,GACGTC,DVA,GACGTCGCG,23.566667,False,None,NaN
5,AatII,GACGTC,DVC,GACGTCTGC,13.400000,False,None,NaN
6,AatII,GACGTC,DVD,GACGTCGAT,23.366667,False,None,NaN
7,AatII,GACGTC,DVE,GACGTCGAA,25.300000,False,None,NaN
8,AatII,GACGTC,DVF,GACGTCTTT,17.300000,False,None,NaN
9,AatII,GACGTC,DVG,GACGTCGGC,21.866667,False,None,NaN



4. Example: Check one enzyme in detail
   Enzyme: AatII
   Total 3mer AAs: 63
   With silent mutation: 0


,enzyme,recognition_site,3mer_aa,optimal_9bp_original,codon_usage_original,can_silent_mutation,optimal_9bp_mutated,codon_usage_mutated
0,AatII,GACGTC,ADV,GCGGACGTC,23.566667,False,None,NaN
1,AatII,GACGTC,ATS,GCGACGTCG,19.333333,False,None,NaN
2,AatII,GACGTC,CDV,TGCGACGTC,13.400000,False,None,NaN
3,AatII,GACGTC,DDV,GATGACGTC,23.366667,False,None,NaN
4,AatII,GACGTC,DVA,GACGTCGCG,23.566667,False,None,NaN
...,...,...,...,...,...,...,...,...
58,AatII,GACGTC,VDV,GTGGACGTC,19.533333,False,None,NaN
59,AatII,GACGTC,VTS,GTGACGTCG,15.300000,False,None,NaN
60,AatII,GACGTC,WDV,TGGGACGTC,14.300000,False,None,NaN
61,AatII,GACGTC,WTS,TGGACGTCG,10.066667,False,None,NaN
